In [1]:
import os

# Tell transformers not to use TensorFlow (we only need PyTorch here)
os.environ["TRANSFORMERS_NO_TF"] = "1"

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import transformers, sys, importlib

print("Python executable:", sys.executable)
print("Transformers version:", transformers.__version__)
print("TensorFlow present?", importlib.util.find_spec("tensorflow") is not None)
print("Keras present?", importlib.util.find_spec("keras") is not None)
print("tf_keras present?", importlib.util.find_spec("tf_keras") is not None)

/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python executable: /Users/chadadelman/anaconda3/envs/chad_env/bin/python
Transformers version: 4.57.3
TensorFlow present? True
Keras present? True
tf_keras present? True


In [2]:
#We will need the training data text summary pairs
%store -r train_paired_summaries


In [3]:
#create object to be passed in as batch
#Needs to be dictinionary of lists
#Will be only training data
text_list=[]
summary_list=[]

for v in train_paired_summaries.values():
    text_list.append(v[0])
    summary_list.append(v[1])

training_batch = dict()
training_batch["text"] = text_list
training_batch["summary"] = summary_list

training_batch_dataset = Dataset.from_dict(training_batch)

In [4]:
# Simple train/validation split
dataset = training_batch_dataset.train_test_split(test_size=0.4, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

train_dataset, eval_dataset

(Dataset({
     features: ['text', 'summary'],
     num_rows: 454
 }),
 Dataset({
     features: ['text', 'summary'],
     num_rows: 303
 }))

In [5]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Tokenizer type:", type(tokenizer))
print("Model type:", type(model))

Tokenizer type: <class 'transformers.models.t5.tokenization_t5_fast.T5TokenizerFast'>
Model type: <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


In [6]:
max_input_length = 64
max_target_length = 32

def preprocess_function(batch):
    # T5 likes a task prefix, e.g. "summarize: "
    inputs = ["summarize: " + t for t in batch["text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        batch["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text", "summary"],
)

tokenized_eval = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text", "summary"],
)

tokenized_train[:2]


Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

{'input_ids': [[21603,
   10,
   10083,
   14407,
   63,
   6,
   205,
   109,
   32,
   102,
   9,
   1313,
   6,
   695,
   32,
   1047,
   3465,
   6,
   20330,
   23,
   152,
   6,
   27,
   52,
   9,
   7,
   6,
   5104,
   9,
   7,
   28,
   717,
   5,
   3,
   9156,
   4170,
   476,
   5,
   216,
   56,
   59,
   2870,
   28,
   140,
   6,
   531,
   1538,
   23,
   302,
   58,
   262,
   7400,
   4882,
   12108,
   3063,
   5,
   465,
   5,
   3,
   9156,
   4170,
   476,
   5,
   1],
  [21603,
   10,
   10083,
   2043,
   7159,
   1395,
   11,
   3,
   9,
   5631,
   6451,
   5,
   309,
   7550,
   21357,
   3205,
   5,
   1263,
   6,
   281,
   6,
   82,
   14620,
   6,
   240,
   3,
   17,
   9492,
   7080,
   173,
   302,
   22,
   4952,
   117,
   18795,
   8,
   2725,
   3,
   849,
   15,
   26,
   12,
   82,
   9360,
   5895,
   7,
   7,
   23,
   26,
   5,
   16344,
   6,
   1011,
   727,
   82,
   313,
   12,
   160,
   2790,
   117,
   8779,
   160,
   27,
   1]],
 'a

In [7]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Data collator ready:", type(data_collator))

Data collator ready: <class 'transformers.data.data_collator.DataCollatorForSeq2Seq'>


In [10]:
#Setting hyperparameters
learning_rate=5e-7
batch_size=2
num_epochs = 8

In [ ]:
# 8. Manual training loop (no Trainer, no Hugging Face Hub)

import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# DataLoader for training
train_dataloader = DataLoader(
    tokenized_train,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for step, batch in enumerate(train_dataloader):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 5 == 0:
            print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f}")

print("Training complete (manual loop).")

Epoch 1, Step 5, Loss: 5.9084
Epoch 1, Step 10, Loss: 4.7879
Epoch 1, Step 15, Loss: 5.6552
Epoch 1, Step 20, Loss: 5.2779
Epoch 1, Step 25, Loss: 4.6201
Epoch 1, Step 30, Loss: 5.0294
Epoch 1, Step 35, Loss: 5.6084
Epoch 1, Step 40, Loss: 4.9782
Epoch 1, Step 45, Loss: 5.0280
Epoch 1, Step 50, Loss: 5.8391
Epoch 1, Step 55, Loss: 3.9301
Epoch 1, Step 60, Loss: 5.2256
Epoch 1, Step 65, Loss: 5.2959
Epoch 1, Step 70, Loss: 5.6186
Epoch 1, Step 75, Loss: 6.6039
Epoch 1, Step 80, Loss: 6.0075
Epoch 1, Step 85, Loss: 4.7645
Epoch 1, Step 90, Loss: 6.2985
Epoch 1, Step 95, Loss: 4.2833
Epoch 1, Step 100, Loss: 5.4962
Epoch 1, Step 105, Loss: 5.1454
Epoch 1, Step 110, Loss: 5.3457
Epoch 1, Step 115, Loss: 6.3349
Epoch 1, Step 120, Loss: 4.4828
Epoch 1, Step 125, Loss: 4.8796
Epoch 1, Step 130, Loss: 5.5652
Epoch 1, Step 135, Loss: 5.4137
Epoch 1, Step 140, Loss: 6.3702
Epoch 1, Step 145, Loss: 6.3312
Epoch 1, Step 150, Loss: 4.9455
Epoch 1, Step 155, Loss: 5.8100
Epoch 1, Step 160, Loss: 5.7

In [9]:
# 9. Quick test generation after manual fine-tuning

model.eval()

test_text = "A prince wrestles with doubt and morality as he considers avenging his father's murder."

inputs = tokenizer(
    "summarize: " + test_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
).to(device)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_length=32,
        num_beams=4,
    )

generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("INPUT:", test_text)
print("SUMMARY:", generated_text)

INPUT: A prince wrestles with doubt and morality as he considers avenging his father's murder.
SUMMARY: prince considers avenging his father's murder. prince wrestles with doubt and morality as he considers avenging
